## <b><font color='darkblue'>Preface</font></b>
([source](https://adk.dev/graphs/)) <font size='3ptx'><b>Graph-based agent workflows in ADK let you build agents with more precise control, creating deterministic processes that combine code logic and AI reasoning capabilities.</b> Graph-based workflows allow you to define your agent logic as a graph of execution nodes and edges, combining AI-powered agent reasoning with deterministic tools and code.</font>
![ui1](https://adk.dev/assets/workflow-design.svg)

<b>Figure 1.</b> A graph-based agent design for flight upgrades, combining workflow nodes of different types, including Functions, human input, Tools, and LLM capabilities.

Prebuilt [**ADK template workflows**](https://adk.dev/agents/workflow-agents/), such as [**Sequential Agents**](https://adk.dev/agents/workflow-agents/sequential-agents/), provide a defined process flow control only across a set of agents. You can continue to build standard ADK agents with long prompts, tools, and use them in graph-based workflow agents.

<b>When you need more precise control, workflow agent graphs give you more flexibility over how tasks are routed and executed</b>. Graph-based workflows provide the following advantages:
- <b>Define precise logic:</b> Explicitly map out routing logic to manage transitions between different nodes.
- <b>Implement complex structures:</b> Build agent workflows that support branching and state management.
- <b>Run chains of functions without AI:</b> Call agent tools and your own code without invoking a generative AI model.
- <b>Enhance reliability</b>: Improve the predictability of your agents by relying on structured node definitions rather than prompts alone.

In [14]:
import asyncio

from google.genai import types
from google.adk import Agent
from google.adk import Workflow
from google.adk import Event
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from pydantic import BaseModel

## <b><font color='darkblue'>Get started</font></b>
<font size='3ptx'><b>This section describes how to get started with graph-based agents</b>. The following example shows how to create a sequential graph-based agent workflow that generates a city name, looks up the current time in that city with code function, and the final agent reports the information.</font>

In [2]:
class CityTime(BaseModel):
    time_info: str  # time information
    city: str       # city name


def lookup_time_function(node_input: str):
    """Simulate returning the current time in the specified city."""
    return CityTime(time_info="10:10 AM", city=node_input)


def get_city_report_root_agent():
    city_generator_agent = Agent(
        name="city_generator_agent",
        model="gemini-flash-latest",
        instruction="""Return the name of a random city.
              Return only the name, nothing else.""",
        output_schema=str,
    )

    city_report_agent = Agent(
        name="city_report_agent",
        model="gemini-flash-latest",
        input_schema=CityTime,
        instruction="""Output following line:
        It is {CityTime.time_info} in {CityTime.city} right now.""",
        output_schema=str,
    )


    def completed_message_function(node_input: str):
        return Event(
            message=f"{node_input}\n WORKFLOW COMPLETED.",
        )

    root_agent = Workflow(
        name="root_agent",
        edges=[
            (
                "START",
                city_generator_agent,
                lookup_time_function,
                city_report_agent,
                completed_message_function,
            )
        ],
    )
    return root_agent

Next, let's prepare the runtime to run the agent:

In [18]:
APP_NAME = "city_report_app"
USER_ID = "1234"
SESSION_ID = "session1234"

# Session and Runner
async def setup_session_and_runner():
  session_service = InMemorySessionService()
  session = await session_service.create_session(
      app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
  runner = Runner(
      agent=get_city_report_root_agent(),
      app_name=APP_NAME,
      session_service=session_service)
  return session, runner


# Agent Interaction
async def call_agent_async(query):
  content = types.Content(role='user', parts=[types.Part(text=query)])
  session, runner = await setup_session_and_runner()
  events = runner.run_async(
      user_id=USER_ID, session_id=SESSION_ID, new_message=content)

  async for event in events:
    if event.is_final_response() and event.content:
      final_response = event.content.parts[0].text
      print("Agent Response: ", final_response)

In [19]:
async def main():
    await call_agent_async('Go')

await main()

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Agent Response:  "Kyoto"
Agent Response:  "It is 10:10 AM in Kyoto right now."
Agent Response:  It is 10:10 AM in Kyoto right now.
 WORKFLOW COMPLETED.


This sample code demonstrates how you can use the <font color='blue'>**Workflow**</font> class to assemble a simple, sequential workflow and alternate between AI agent processing and code execution.

While you could perform these steps using a single agent with a longer prompt and a tool call, <b>the graph-based approach gives you precise control over the task execution order and the data output from each step</b>.

For more information about data handling with graph-based workflows, see [**Data handling with workflow nodes and agents**](https://adk.dev/graphs/data-handling/).

## <b><font color='darkblue'>Build processes with graphs</font></b>
<font size='3ptx'>You can use prompt-based agents to define multiple step processes with descriptions of tasks and procedures using the instructions field of an ADK agent. <b>However, as your instructions and procedures become longer and more complicated, making sure that the agent is following each step and guideline becomes more complicated and less reliable.</b></font>

<b>Graph-based workflow agents provide a significant advantage over prompt-based agents by allowing you to specifically define the overall process workflow in code.</b> With graph-based agent workflows, each step of the process can be defined as an execution <b><font color='darkblue'>Node</font></b> in a graph and each node can be an AI agent, Tool, or your programmed code.

The following diagram illustrates how a simple prompt-based agent would translate into a workflow agent graph:
![image2](https://adk.dev/assets/prompts-to-graphs.svg)

<b>Figure 2.</b> Structure of prompt-based agent instructions translated into a graph-based workflow.

<b>Moving from prompt-based agents to graph-based workflow agents allows you to explicitly break out the tasks of a procedure to define a specific execution flow.</b> Once defined, the agent application flows the steps in the graph, switching between non-deterministic AI-powered agents and deterministic code as needed.

The following code sample shows how the workflow graph in Figure 2 could be translated into a graph-based agent using the <font color='blue'>**Workflow**</font> class:

In [20]:
def response_1_bug():
    return Event(message="Handling bug...")


def response_2_support():
    return Event(message="Handling customer support...")


def response_3_logistics():
    return Event(message="Handling logistics...")


def router(node_input: str):
    routes = node_input.split(",")
    routes = [route.strip() for route in routes]
    return Event(route=routes)


def get_message_process_root_agent():
    process_message = Agent(
        name="process_message",
        model="gemini-flash-latest",
        instruction="""Classify user message into either "BUG", "CUSTOMER_SUPPORT",
              or "LOGISTICS". If you think a message applies to more than one category,
              reply with a comma separated list of categories.
        """,
        output_schema=str,
    )

    root_agent = Workflow(
       name="routing_workflow",
       edges=[
           ("START", process_message, router),
           ( router,
               {
                   "BUG": response_1_bug,
                   "CUSTOMER_SUPPORT": response_2_support,
                   "LOGISTICS": response_3_logistics,
               }
           )
       ],
    )
    return root_agent

Time to test the code:

In [21]:
APP_NAME = "message_process_app"
USER_ID = "1234"
SESSION_ID = "session1234"

# Session and Runner
async def setup_message_process_session_and_runner():
  session_service = InMemorySessionService()
  session = await session_service.create_session(
      app_name=APP_NAME, user_id=USER_ID, session_id=SESSION_ID)
  runner = Runner(
      agent=get_message_process_root_agent(),
      app_name=APP_NAME,
      session_service=session_service)
  return session, runner


# Agent Interaction
async def call_message_agent_async(query):
  content = types.Content(role='user', parts=[types.Part(text=query)])
  session, runner = await setup_message_process_session_and_runner()
  events = runner.run_async(
      user_id=USER_ID, session_id=SESSION_ID, new_message=content)

  async for event in events:
    if event.is_final_response() and event.content:
      final_response = event.content.parts[0].text
      print("Agent Response: ", final_response)

In [23]:
await call_message_agent_async("This is a bug")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Agent Response:  "BUG"
Agent Response:  Handling bug...


In [26]:
await call_message_agent_async("This is a customer support ticket")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Agent Response:  "CUSTOMER_SUPPORT"
Agent Response:  Handling customer support...


In [27]:
await call_message_agent_async("A logistics request.")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


Agent Response:  "LOGISTICS"
Agent Response:  Handling logistics...


<b>This sample code demonstrates how you can use an edges array to define a graph with routes between a set of nodes, which are discrete tasks that can include agents, Tools, your code, and even additional <font color='blue'>Workflows</font></b>. For information about building advanced graphs for workflows, see [**Build graph routes for workflow agents**](https://adk.dev/graphs/routes/).

## <b><font color='darkblue'>Known limitations</font></b>
There are some known limitations with graph-based workflows. They are not compatible with the following ADK features:
- <b>Live Streaming functionality</b> is not compatible with graph-based workflows.
- <b>Integrations</b>: Some third-party Integrations may not be compatible with graph-based workflows.